# End-to-End Trace: Research Agent + Cortex

Traces one real research episode through every stage (Cortex retrieve → Qwen config →
GPT Researcher → Grader/Reviser → Reflection → Cortex write-back), then runs a **second**
related episode to prove Cortex memory actually gets *used* on retrieval, not just written.

Requires a running Ollama server with a Qwen model pulled (confirmed working with `qwen3:14b`).
Each research episode is a real multi-minute run on a local model — this notebook makes real
calls, nothing here is mocked or dry-run.

In [ ]:
import sys, os, asyncio, json, sqlite3
from pathlib import Path

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "cortex").exists():
    REPO_ROOT = REPO_ROOT.parent  # in case the notebook is opened from a subdirectory
sys.path.insert(0, str(REPO_ROOT))

# Qwen via local Ollama -- adjust QWEN_MODEL if you've pulled a different one (`ollama list`)
os.environ["QWEN_API_KEY"] = "ollama"
os.environ["QWEN_BASE_URL"] = "http://localhost:11434/v1"
os.environ["QWEN_MODEL"] = "qwen3:14b"
os.environ["RETRIEVER"] = "duckduckgo"

# Fresh, isolated Cortex store + workspace for this trace, so it's reproducible
# and doesn't mix with any other run's memory.
TRACE_DIR = REPO_ROOT / "notebook_trace_data"
CORTEX_DIR = str(TRACE_DIR / "cortex_data")
WORKSPACE_DIR = str(TRACE_DIR / "workspace")
print("Repo root:", REPO_ROOT)
print("Cortex dir:", CORTEX_DIR)


## Stage 1 — Cortex retrieve (cold start)

First episode, nothing in memory yet — this should come back empty.

In [ ]:
from cortex import CortexStore, MemoryGraph, retrieve

QUESTION_1 = "What is the boiling point of water in Celsius at sea level?"

store = CortexStore(persist_dir=CORTEX_DIR)
graph = MemoryGraph(persist_path=str(Path(CORTEX_DIR) / "graph.sqlite3"))

retrieved_1 = retrieve(store, graph, QUESTION_1, scope="default")
print(retrieved_1.to_prompt_text())


## Stage 2 — Qwen configuration

Confirms which provider path got selected (`dashscope` vs `vllm_openai` vs `dry_run`) before any real call happens.

In [ ]:
from agents.research import configure_qwen

provider = configure_qwen()
print("Provider selected:", provider)
assert provider != "dry_run", "QWEN_API_KEY/QWEN_BASE_URL not picked up -- check Ollama is running"
print("FAST_LLM   =", os.environ.get("FAST_LLM"))
print("SMART_LLM  =", os.environ.get("SMART_LLM"))
print("STRATEGIC_LLM =", os.environ.get("STRATEGIC_LLM"))


## Stage 3 — Fold memory into the query, run GPT Researcher

`context=` on the GPTResearcher constructor gets overwritten by `conduct_research()` (confirmed by reading `agent.py`), so memory gets folded into the query text itself -- the one injection point every internal stage actually reads.

In [ ]:
from gpt_researcher import GPTResearcher

memory_context_1 = retrieved_1.to_prompt_text()
research_query_1 = QUESTION_1
if "no prior memory" not in memory_context_1:
    research_query_1 = (
        f"{QUESTION_1}\n\nRelevant context from prior research episodes -- do not "
        f"repeat work already covered, and treat Caution entries as known dead "
        f"ends:\n{memory_context_1}"
    )
print("Research query sent to GPT Researcher:\n")
print(research_query_1)


In [ ]:
researcher_1 = GPTResearcher(query=research_query_1, report_type="research_report")
await researcher_1.conduct_research()
print("conduct_research() done.")


In [ ]:
draft_1 = await researcher_1.write_report()
print(f"Draft length: {len(draft_1)} chars\n")
print(draft_1[:1500], "...\n[truncated for display]")


## Stage 4 — Grader

Independent check: does the draft actually support every claim, address the question, and cite real sources?

In [ ]:
from agents.research.reflection import grade

verdict_1, g_in, g_out = await grade(QUESTION_1, draft_1)
print("Verdict:", verdict_1.verdict)
print("Mistakes found:", verdict_1.mistakes or "(none)")
print(f"Grader call cost: {g_in} in / {g_out} out tokens")


## Stage 4b — Reviser loop (only runs if the Grader asked for a revision)

Mirrors `research_pipeline.py`'s actual loop, including the hard policy cap that exists independently of `MAX_REVISIONS`.

In [ ]:
from agents.research.reflection import revise
from agents.research.policies import DEFAULT_POLICIES, PolicyState, evaluate, Decision

MAX_REVISIONS = 3
policy_state = PolicyState()
healer_attempts = []
revision_n = 0
verdict = verdict_1
draft_1_final = draft_1
total_in, total_out = g_in, g_out

while verdict.verdict != "pass" and revision_n < MAX_REVISIONS:
    policy_result = evaluate(DEFAULT_POLICIES, policy_state)
    if policy_result.decision in (Decision.DENY, Decision.ASK):
        print("Policy stopped the loop:", policy_result.reason)
        break
    print(f"--- Revision {revision_n + 1} ---")
    draft_1_final, r_in, r_out = await revise(draft_1_final, verdict.mistakes)
    total_in += r_in; total_out += r_out
    healer_attempts.append({"mistakes": verdict.mistakes, "revision": revision_n + 1})
    verdict, g_in, g_out = await grade(QUESTION_1, draft_1_final)
    total_in += g_in; total_out += g_out
    print("New verdict:", verdict.verdict, "| mistakes:", verdict.mistakes or "(none)")
    revision_n += 1
    policy_state.revision_count = revision_n
    policy_state.total_input_tokens, policy_state.total_output_tokens = total_in, total_out

print(f"\nFinal verdict: {verdict.verdict} after {revision_n} revision(s)")


## Stage 5 — Reflection

Produces the three fields Cortex actually persists for future episodes to learn from.

In [ ]:
from agents.research.reflection import reflect
from cortex import Outcome

outcome_1 = Outcome.SUCCESS if verdict.verdict == "pass" else (
    Outcome.PARTIAL if revision_n > 0 else Outcome.FAILURE
)
summary_text_1 = f"research: {draft_1_final[:150]}\ngrade: final_verdict={verdict.verdict} revisions={revision_n}"
reflection_1, r_in, r_out = await reflect(QUESTION_1, outcome_1.value, summary_text_1)
total_in += r_in; total_out += r_out

print("Outcome:", outcome_1.value)
print("abstracted_pattern:", reflection_1.abstracted_pattern)
print("failure_summary:", reflection_1.failure_summary)
print("distilled_learnings:", reflection_1.distilled_learnings)


## Stage 6 — Cortex write-back

Persists the episode. `recall_count`/`intuition_count` should go from 0 to 1.

In [ ]:
from cortex import Episode, write_back
from cortex.schema import ActionArcStep, CostActual, HealerAttempt

print("Cortex stats BEFORE write-back:", store.stats())

episode_1 = Episode(
    domain="research",
    scope="default",
    problem_summary=QUESTION_1,
    action_arc=[ActionArcStep(action="research", tool_used="GPTResearcher", output=draft_1_final[:300])],
    healer_attempts=[HealerAttempt(hypothesis="; ".join(h["mistakes"]) or "unspecified",
                                    fix=f"revision {h['revision']}", outcome="revise") for h in healer_attempts],
    outcome=outcome_1,
    final_solution=draft_1_final if outcome_1 != Outcome.FAILURE else None,
    abstracted_pattern=reflection_1.abstracted_pattern,
    failure_summary=reflection_1.failure_summary,
    distilled_learnings=reflection_1.distilled_learnings,
    cost_actual=CostActual(input_tokens=total_in, output_tokens=total_out),
    healer_attempt_count=revision_n,
)

write_back(store, graph, episode_1, retrieved=retrieved_1)
print("\nCortex stats AFTER write-back:", store.stats())


## Stage 7 — Inspect the SQLite-backed memory graph directly

Proves persistence is real, queryable disk state, not an in-memory illusion.

In [ ]:
graph_db_path = Path(CORTEX_DIR) / "graph.sqlite3"
print("Graph DB file exists:", graph_db_path.exists(), "--", graph_db_path)

conn = sqlite3.connect(graph_db_path)
print("\nNodes:")
for row in conn.execute("SELECT node_id, node_type FROM nodes"):
    print(" ", row)
print("\nEdges:")
for row in conn.execute("SELECT source, target, edge_type, weight FROM edges"):
    print(" ", row)
conn.close()


## Stage 8 — Second episode: does memory actually get *used*?

A related question, close enough that Recall/Intuition should surface episode 1. This is the real test of "memory," not just "logging."

In [ ]:
QUESTION_2 = "At what temperature does water boil, measured in Celsius, under normal atmospheric pressure?"

retrieved_2 = retrieve(store, graph, QUESTION_2, scope="default")
print(retrieved_2.to_prompt_text())
print("\nRecall hits:", len(retrieved_2.recall_hits))
print("Intuition hits:", len(retrieved_2.intuition_hits))


If Recall or Intuition surfaced episode 1 above, memory is doing real work across episodes, not just accumulating a write-only log. Run the full pipeline for question 2 the same way as Stage 3 onward if you want the complete second trace, or use `research_pipeline.run_research_pipeline(QUESTION_2, cortex_dir=CORTEX_DIR, workspace_dir=WORKSPACE_DIR)` for the one-shot version of everything above.

## What this notebook does *not* cover

- **RSI (Spec 5)**: `rsi/` and `selfmod/` are separate, slow, multi-minute-per-cycle
  processes (a harnessing-level cycle benchmarks 5 tasks twice — baseline and candidate).
  Run them directly: `python cli.py --self-improve --harness research`.
- **The coding harness**: `pipeline.py` / `agents/base.py` / `agents/pipeline_agents.py`,
  Anthropic-backed, untouched by anything in this notebook.
- **Policy DENY in practice**: the revision loop above includes the policy check, but
  the default caps (6 revisions, 900s, 500k tokens) are generous enough that a normal
  run won't trigger one — see `agents/research/policies.py` to lower them if you want
  to see a DENY fire.